In [3]:
# ================================================================
# REVIEWER 1 / PROBLEM 2
# E5-LARGE-INSTRUCT
# INSTRUCTION-AWARE NON-FT STRICT-FAIR
# FULL RECOMPUTATION DIRECTLY FROM THE TWO JSON FILES
# ================================================================
#
# REQUIRED INPUT FILES:
#   baseline_15000.json
#   kazakh_segmented_15000.json
#
# NO FINE-TUNING
#
# Protocol:
#   strict-fair exact CLEAN-question intersection = 6,759
#   seed = 42
#   train = 6,083
#   test  = 676
#
# BASE:
#   clean question -> nearest CLEAN training question
#
# SEG:
#   morph-marked question -> nearest morph-marked training question
#
# SAME CLEAN answer candidate/reference space in BOTH conditions.
#
# E5 formatting:
#   Query:
#     Instruct: Given a question in Kazakh, retrieve the most relevant answer.
#     Query: <question>
#
#   Answers/passages:
#     plain CLEAN answer text
#
# Metrics:
#   Exact@1
#   TokenF1@1
#   MeanCos@1 (QSim)
#   Semantic@1 (answer cosine >= 0.85)
#   BERTScoreF1@1
#
# Statistics:
#   Continuous metrics:
#       paired two-sided sign-flip permutation, 10,000
#   Binary metrics:
#       exact two-sided McNemar
#   95% CI:
#       paired bootstrap, 10,000
#   Multiple testing:
#       Holm correction across 5 metrics
#
# Delta = SEG - BASE
# ================================================================


# ================================================================
# 0. ENVIRONMENT
# ================================================================

import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True"
)

import sys
import re
import json
import glob
import random
import hashlib
import subprocess
import zipfile
from pathlib import Path
from typing import List, Dict, Any, Optional


# ================================================================
# 1. INSTALL / IMPORT
# ================================================================

def ensure(pkg, import_name=None):
    name = import_name or pkg.replace("-", "_")
    try:
        __import__(name)
    except Exception:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pkg
            ]
        )


ensure("numpy")
ensure("pandas")
ensure("scipy")
ensure("scikit-learn", "sklearn")
ensure(
    "sentence-transformers",
    "sentence_transformers"
)
ensure(
    "bert-score",
    "bert_score"
)


import numpy as np
import pandas as pd
import torch

from scipy.stats import binomtest
from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer
from bert_score import score as bert_score


try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.6f}"
)


# ================================================================
# 2. CONFIG
# ================================================================

BASE_FILENAME = "baseline_15000.json"
SEG_FILENAME  = "kazakh_segmented_15000.json"

MODEL_NAME = (
    "intfloat/"
    "multilingual-e5-large-instruct"
)

SEED = 42
TEST_SIZE = 0.10

SEM_THR = 0.85

N_BOOT = 10_000
N_PERM = 10_000
STAT_ALPHA = 0.05
STAT_SEED = 20260901

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

ENCODE_BATCH = (
    32
    if torch.cuda.is_available()
    else 4
)

BERTSCORE_BATCH = (
    16
    if torch.cuda.is_available()
    else 4
)

EXPECTED = {
    "base_records": 14991,
    "seg_records": 14998,
    "base_unique": 14689,
    "seg_unique": 14696,
    "paired": 6759,
    "train": 6083,
    "test": 676,
}

EXPECTED_SPLIT_SHA256 = (
    "867d3305fbc71068afedeae71b4ef102"
    "21968d23270202bf0eda860fc8bf880b"
)

OUT_DIR = Path(
    "/content/"
    "e5_instructionaware_nonft_recomputed"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. REPRODUCIBILITY
# ================================================================

def set_all_seeds(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print("=" * 78)
print(
    "E5-LARGE-INSTRUCT "
    "NON-FT STRICT-FAIR RECOMPUTATION"
)
print("=" * 78)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print(
        "⚠️ GPU not detected. "
        "The code can run on CPU, "
        "but E5-large will be slow."
    )

print("Model:", MODEL_NAME)

print(
    "E5 query formatting: "
    "instruction-aware"
)

print(
    "Answer/passages: "
    "plain CLEAN text"
)


# ================================================================
# 4. FIND INPUT FILES
# ================================================================

def find_data_path(filename):

    candidates = [
        filename,
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]

    for p in candidates:

        if Path(p).is_file():
            return str(
                Path(p).resolve()
            )

    hits = glob.glob(
        f"/content/**/{filename}",
        recursive=True
    )

    hits += glob.glob(
        f"./**/{filename}",
        recursive=True
    )

    hits = [
        h
        for h in hits
        if Path(h).is_file()
    ]

    if hits:
        return str(
            Path(hits[0]).resolve()
        )

    return ""


base_path = find_data_path(
    BASE_FILENAME
)

seg_path = find_data_path(
    SEG_FILENAME
)


# ================================================================
# 5. UPLOAD IF FILES ARE MISSING
# ================================================================

if (
    not base_path
    or not seg_path
):

    if IN_COLAB:

        print(
            "\n"
            "======================================================\n"
            "UPLOAD THESE TWO FILES:\n"
            "1. baseline_15000.json\n"
            "2. kazakh_segmented_15000.json\n"
            "======================================================"
        )

        files.upload()

        base_path = find_data_path(
            BASE_FILENAME
        )

        seg_path = find_data_path(
            SEG_FILENAME
        )


if (
    not base_path
    or not seg_path
):

    raise FileNotFoundError(
        "Could not find both JSON files."
    )


# ================================================================
# 6. ROBUST JSON LOADER
# ================================================================

def normalize_records(data):

    if not isinstance(data, list):
        raise ValueError(
            "Parsed QA data must be a list."
        )

    out = []

    for x in data:

        if not isinstance(x, dict):
            continue

        q = (
            x.get("question")
            or x.get("instruction")
            or ""
        )

        a = (
            x.get("answer")
            or x.get("response")
            or ""
        )

        q = str(q).strip()
        a = str(a).strip()

        if q and a:

            out.append(
                {
                    "question": q,
                    "answer": a,
                }
            )

    if not out:
        raise ValueError(
            "No valid QA records found."
        )

    return out


def load_qa_records(path):

    text = Path(path).read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    if not text:
        raise ValueError(
            f"Empty file: {path}"
        )

    # ------------------------
    # Standard JSON array
    # ------------------------

    if text.startswith("["):

        try:

            return normalize_records(
                json.loads(text)
            )

        except Exception:
            pass

    # ------------------------
    # JSONL
    # ------------------------

    lines = [
        x.strip().rstrip(",")
        for x in text.splitlines()
        if x.strip()
    ]

    if (
        lines
        and lines[0].startswith("{")
    ):

        try:

            return normalize_records(
                [
                    json.loads(x)
                    for x in lines
                ]
            )

        except Exception:
            pass

    # ------------------------
    # Robust object scanner
    # ------------------------

    objs = []

    buf = []

    depth = 0

    in_string = False

    escaped = False

    started = False


    for ch in text:

        if not started:

            if ch == "{":

                started = True
                depth = 1
                buf = ["{"]

            continue


        buf.append(ch)


        if in_string:

            if escaped:
                escaped = False

            elif ch == "\\":
                escaped = True

            elif ch == '"':
                in_string = False

        else:

            if ch == '"':
                in_string = True

            elif ch == "{":
                depth += 1

            elif ch == "}":

                depth -= 1

                if depth == 0:

                    s = "".join(buf)

                    started = False
                    buf = []

                    try:
                        objs.append(
                            json.loads(s)
                        )
                    except Exception:
                        pass


    return normalize_records(objs)


base_rows = load_qa_records(
    base_path
)

seg_rows = load_qa_records(
    seg_path
)


print(
    "\n================ FILES / RAW COUNTS ================"
)

print(
    f"BASE: {base_path} | "
    f"N={len(base_rows):,}"
)

print(
    f"SEG : {seg_path} | "
    f"N={len(seg_rows):,}"
)


# ================================================================
# 7. STRICT-FAIR TEXT NORMALIZATION
# ================================================================

_punct_space_left = re.compile(
    r"\s+([.,!?;:%)\]\}])"
)

_punct_space_right = re.compile(
    r"([(\[\{])\s+"
)

_multi_space = re.compile(
    r"\s+"
)


def norm_space_punct(text):

    text = str(text)

    text = text.replace(
        " - ",
        "-"
    )

    text = _punct_space_left.sub(
        r"\1",
        text
    )

    text = _punct_space_right.sub(
        r"\1",
        text
    )

    text = _multi_space.sub(
        " ",
        text
    ).strip()

    return text


def morph_marker_view(text):

    if text is None:
        text = ""

    return norm_space_punct(
        str(text)
    )


def clean_view(text):

    if text is None:
        text = ""

    text = str(text)

    text = text.replace(
        "@@ ",
        ""
    )

    text = text.replace(
        "@@",
        ""
    )

    return norm_space_punct(
        text
    )


def norm_for_exact(text):

    return re.sub(
        r"\s+",
        " ",
        norm_space_punct(
            str(text)
        ).lower()
    ).strip()


# ================================================================
# 8. TOKEN F1
# ================================================================

def tokens(text):

    text = norm_space_punct(
        str(text)
    ).lower()

    return re.findall(
        r"[a-zA-Zа-яА-Я"
        r"әғқңөұүһі"
        r"ӘҒҚҢӨҰҮҺІ"
        r"0-9]+",
        text
    )


def token_f1(pred, gold):

    from collections import Counter

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    pc = Counter(p)
    gc = Counter(g)

    inter = sum(
        (pc & gc).values()
    )

    if inter == 0:
        return 0.0

    precision = (
        inter / len(p)
    )

    recall = (
        inter / len(g)
    )

    return float(
        2
        * precision
        * recall
        /
        (
            precision
            + recall
            + 1e-12
        )
    )


# ================================================================
# 9. BUILD EXACT STRICT-FAIR PAIRED SET
# ================================================================

def first_by_clean_key(rows):

    result = {}

    for row in rows:

        key = clean_view(
            row["question"]
        )

        if (
            key
            and key not in result
        ):

            result[key] = row

    return result


base_map = first_by_clean_key(
    base_rows
)

seg_map = first_by_clean_key(
    seg_rows
)


common_keys = sorted(
    set(base_map)
    &
    set(seg_map)
)


paired = []


for pair_id, key in enumerate(
    common_keys
):

    b = base_map[key]

    s = seg_map[key]

    paired.append(
        {
            "pair_id": pair_id,

            "pair_key": key,

            "base_q":
                clean_view(
                    b["question"]
                ),

            "seg_q_clean":
                clean_view(
                    s["question"]
                ),

            "seg_q_morph":
                morph_marker_view(
                    s["question"]
                ),

            # IMPORTANT:
            # SAME CLEAN baseline answer
            # for BOTH conditions
            "base_a":
                norm_space_punct(
                    b["answer"]
                ),
        }
    )


print(
    "\n================ STRICT PAIRING AUDIT ================"
)

print(
    f"Clean records                  = "
    f"{len(base_rows):,}"
)

print(
    f"Segmented records              = "
    f"{len(seg_rows):,}"
)

print(
    f"Clean unique keys              = "
    f"{len(base_map):,}"
)

print(
    f"Segmented unique keys          = "
    f"{len(seg_map):,}"
)

print(
    f"Strict-fair paired keys        = "
    f"{len(paired):,}"
)

print(
    f"Clean duplicates collapsed     = "
    f"{len(base_rows)-len(base_map):,}"
)

print(
    f"Segmented duplicates collapsed = "
    f"{len(seg_rows)-len(seg_map):,}"
)


actual = {
    "base_records": len(base_rows),
    "seg_records": len(seg_rows),
    "base_unique": len(base_map),
    "seg_unique": len(seg_map),
    "paired": len(paired),
}


wrong = {}


for k in [
    "base_records",
    "seg_records",
    "base_unique",
    "seg_unique",
    "paired",
]:

    if actual[k] != EXPECTED[k]:

        wrong[k] = (
            actual[k],
            EXPECTED[k]
        )


if wrong:

    raise RuntimeError(
        "STOP: source/pairing counts differ "
        f"from expected protocol: {wrong}"
    )


print(
    "✅ Strict-fair pairing counts reproduced exactly."
)


# ================================================================
# 10. EXACT SAME TRAIN / TEST SPLIT
# ================================================================

train_rows, test_rows = train_test_split(
    paired,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)


if (
    len(train_rows)
    != EXPECTED["train"]
    or
    len(test_rows)
    != EXPECTED["test"]
):

    raise RuntimeError(
        "STOP: expected "
        "6083 train / 676 test, got "
        f"{len(train_rows)} / "
        f"{len(test_rows)}"
    )


train_keys = sorted(
    x["pair_key"]
    for x in train_rows
)

test_keys = sorted(
    x["pair_key"]
    for x in test_rows
)


split_hash = hashlib.sha256(
    (
        "\n".join(train_keys)
        +
        "\n---TEST---\n"
        +
        "\n".join(test_keys)
    ).encode("utf-8")
).hexdigest()


print(
    "\n================ STRICT-FAIR SPLIT ================"
)

print(
    f"Total paired = {len(paired):,}"
)

print(
    f"Train        = {len(train_rows):,}"
)

print(
    f"Test         = {len(test_rows):,}"
)

print(
    f"Seed         = {SEED}"
)

print(
    f"Split SHA256 = {split_hash}"
)


if (
    split_hash
    != EXPECTED_SPLIT_SHA256
):

    raise RuntimeError(
        "STOP: split differs from "
        "MiniLM/BGE/E5 strict-fair split.\n"
        f"Obtained: {split_hash}\n"
        f"Expected: {EXPECTED_SPLIT_SHA256}"
    )


print(
    "✅ Exact same 6083/676 split reproduced."
)

print(
    "✅ CLEAN candidate/gold answers "
    "used in BOTH conditions."
)


# ================================================================
# 11. SAVE SPLIT MANIFEST
# ================================================================

train_key_set = set(
    train_keys
)


split_manifest = []


for row in paired:

    split_manifest.append(
        {
            "pair_id":
                row["pair_id"],

            "pair_key":
                row["pair_key"],

            "split":
                (
                    "train"
                    if row["pair_key"]
                    in train_key_set
                    else "test"
                ),
        }
    )


pd.DataFrame(
    split_manifest
).to_csv(
    OUT_DIR /
    "strict_fair_split_seed42.csv",

    index=False,

    encoding="utf-8-sig"
)


# ================================================================
# 12. E5-LARGE-INSTRUCT INPUT FORMAT
# ================================================================

E5_QUERY_INSTRUCTION = (
    "Instruct: Given a question in Kazakh, "
    "retrieve the most relevant answer.\n"
    "Query: "
)


def e5_wrap_query(text):

    return (
        E5_QUERY_INSTRUCTION
        +
        str(text)
    )


def encode_e5_query(
    model,
    texts
):

    wrapped = [
        e5_wrap_query(t)
        for t in texts
    ]

    return model.encode(
        wrapped,
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def encode_e5_passage(
    model,
    texts
):

    return model.encode(
        texts,
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


# ================================================================
# 13. LOAD PRETRAINED E5 — NO FINE-TUNING
# ================================================================

print(
    "\n"
    + "=" * 78
)

print(
    "LOADING PRETRAINED "
    "MULTILINGUAL-E5-LARGE-INSTRUCT"
)

print(
    "=" * 78
)


set_all_seeds(SEED)


model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)


print(
    "✅ Pretrained model loaded."
)

print(
    "❗ Fine-tuning is NOT performed."
)


# ================================================================
# 14. EVALUATION FUNCTION
# ================================================================

def evaluate_condition(
    model,
    train_rows,
    test_rows,
    condition
):

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"EVALUATING NON-FT {condition}"
    )

    print(
        "=" * 78
    )


    # ------------------------
    # Training/candidate questions
    # ------------------------

    if condition == "BASE":

        index_questions = [
            r["base_q"]
            for r in train_rows
        ]

        test_questions = [
            r["base_q"]
            for r in test_rows
        ]

    elif condition == "SEG":

        index_questions = [
            r["seg_q_morph"]
            for r in train_rows
        ]

        test_questions = [
            r["seg_q_morph"]
            for r in test_rows
        ]

    else:

        raise ValueError(
            "condition must be BASE or SEG"
        )


    # SAME CLEAN answer space
    candidate_answers = [
        r["base_a"]
        for r in train_rows
    ]

    gold_answers = [
        r["base_a"]
        for r in test_rows
    ]


    print(
        "Encoding training questions..."
    )

    index_q_emb = encode_e5_query(
        model,
        index_questions
    )


    print(
        "Encoding candidate CLEAN answers..."
    )

    candidate_a_emb = encode_e5_passage(
        model,
        candidate_answers
    )


    print(
        "Encoding test questions..."
    )

    test_q_emb = encode_e5_query(
        model,
        test_questions
    )


    print(
        "Encoding gold CLEAN answers..."
    )

    gold_a_emb = encode_e5_passage(
        model,
        gold_answers
    )


    details = []


    for i, row in enumerate(
        test_rows
    ):

        # Embeddings are L2 normalized,
        # therefore dot product = cosine.
        sims = np.dot(
            index_q_emb,
            test_q_emb[i]
        )

        j = int(
            np.argmax(sims)
        )


        qsim = float(
            sims[j]
        )


        pred_answer = (
            candidate_answers[j]
        )

        gold_answer = (
            gold_answers[i]
        )


        exact = float(
            norm_for_exact(
                pred_answer
            )
            ==
            norm_for_exact(
                gold_answer
            )
        )


        tf1 = token_f1(
            pred_answer,
            gold_answer
        )


        ans_cos = float(
            np.dot(
                candidate_a_emb[j],
                gold_a_emb[i]
            )
        )


        sem_hit = float(
            ans_cos >= SEM_THR
        )


        details.append(
            {
                "stage":
                    "NON_FT",

                "condition":
                    condition,

                "pair_id":
                    row["pair_id"],

                "pair_key":
                    row["pair_key"],

                "test_question":
                    test_questions[i],

                "gold_answer":
                    gold_answer,

                "pred_answer":
                    pred_answer,

                "retrieved_train_question":
                    index_questions[j],

                "retrieved_train_row":
                    j,

                "QSim":
                    qsim,

                "Exact":
                    exact,

                "TokenF1":
                    tf1,

                "AnsCos":
                    ans_cos,

                "SemHit":
                    sem_hit,
            }
        )


    return details


# ================================================================
# 15. RUN BASE AND SEG
# ================================================================

base_details = evaluate_condition(
    model,
    train_rows,
    test_rows,
    "BASE"
)


seg_details = evaluate_condition(
    model,
    train_rows,
    test_rows,
    "SEG"
)


# ================================================================
# 16. PAIRING VALIDATION
# ================================================================

if (
    len(base_details) != 676
    or
    len(seg_details) != 676
):

    raise RuntimeError(
        "STOP: expected 676/676 "
        "item-level outputs."
    )


for i, (b, s) in enumerate(
    zip(
        base_details,
        seg_details
    )
):

    if (
        b["pair_id"]
        != s["pair_id"]
    ):

        raise RuntimeError(
            f"pair_id mismatch at item {i}"
        )


    if (
        b["pair_key"]
        != s["pair_key"]
    ):

        raise RuntimeError(
            f"pair_key mismatch at item {i}"
        )


    if (
        norm_for_exact(
            b["gold_answer"]
        )
        !=
        norm_for_exact(
            s["gold_answer"]
        )
    ):

        raise RuntimeError(
            f"gold-answer mismatch "
            f"at item {i}"
        )


print(
    "\n✅ BASE and SEG paired outputs "
    "validated for all 676 items."
)


# ================================================================
# 17. FREE E5 GPU MEMORY BEFORE BERTSCORE
# ================================================================

del model

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ================================================================
# 18. BERTSCORE — SAME GOLD ANSWERS
# ================================================================

print(
    "\n"
    + "=" * 78
)

print(
    "COMPUTING BERTSCORE"
)

print(
    "=" * 78
)


base_predictions = [
    r["pred_answer"]
    for r in base_details
]

seg_predictions = [
    r["pred_answer"]
    for r in seg_details
]

gold_answers = [
    r["gold_answer"]
    for r in base_details
]


bert_lang = "kk"


try:

    _, _, base_f1 = bert_score(
        base_predictions,
        gold_answers,
        lang=bert_lang,
        rescale_with_baseline=True,
        batch_size=BERTSCORE_BATCH,
        verbose=False,
    )


    _, _, seg_f1 = bert_score(
        seg_predictions,
        gold_answers,
        lang=bert_lang,
        rescale_with_baseline=True,
        batch_size=BERTSCORE_BATCH,
        verbose=False,
    )


except Exception as e:

    print(
        "\n⚠️ BERTScore with lang='kk' "
        "failed:"
    )

    print(repr(e))

    print(
        "\nTrying lang='tr'..."
    )

    bert_lang = "tr"


    _, _, base_f1 = bert_score(
        base_predictions,
        gold_answers,
        lang=bert_lang,
        rescale_with_baseline=True,
        batch_size=BERTSCORE_BATCH,
        verbose=False,
    )


    _, _, seg_f1 = bert_score(
        seg_predictions,
        gold_answers,
        lang=bert_lang,
        rescale_with_baseline=True,
        batch_size=BERTSCORE_BATCH,
        verbose=False,
    )


base_f1 = (
    base_f1
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)

seg_f1 = (
    seg_f1
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)


if (
    len(base_f1) != 676
    or
    len(seg_f1) != 676
):

    raise RuntimeError(
        "Unexpected BERTScore output length."
    )


for i in range(676):

    base_details[i][
        "BERTScoreF1"
    ] = float(
        base_f1[i]
    )

    seg_details[i][
        "BERTScoreF1"
    ] = float(
        seg_f1[i]
    )


print(
    f"✅ BERTScore language used: "
    f"{bert_lang}"
)


# ================================================================
# 19. SUMMARY FUNCTION
# ================================================================

def metric_summary(details):

    return {

        "Exact@1":
            float(
                np.mean(
                    [
                        r["Exact"]
                        for r in details
                    ]
                )
            ),

        "TokenF1@1":
            float(
                np.mean(
                    [
                        r["TokenF1"]
                        for r in details
                    ]
                )
            ),

        "MeanCos@1(QSim)":
            float(
                np.mean(
                    [
                        r["QSim"]
                        for r in details
                    ]
                )
            ),

        "Semantic@1(ans_cos≥0.85)":
            float(
                np.mean(
                    [
                        r["SemHit"]
                        for r in details
                    ]
                )
            ),

        "BERTScoreF1@1":
            float(
                np.mean(
                    [
                        r["BERTScoreF1"]
                        for r in details
                    ]
                )
            ),
    }


base_summary = metric_summary(
    base_details
)

seg_summary = metric_summary(
    seg_details
)


summary_df = pd.DataFrame(
    [
        {
            "Stage":
                "NON_FT",

            "Condition":
                "BASE_CLEAN",

            "N":
                676,

            **base_summary,
        },

        {
            "Stage":
                "NON_FT",

            "Condition":
                "SEG_MORPH_ONLY",

            "N":
                676,

            **seg_summary,
        },
    ]
)


print(
    "\n\n"
    + "=" * 78
)

print(
    "TABLE 1. E5 INSTRUCTION-AWARE "
    "NON-FT STRICT-FAIR RESULTS"
)

print(
    "=" * 78
)

print(
    summary_df.to_string(
        index=False
    )
)


# ================================================================
# 20. SANITY CHECK AGAINST THE PREVIOUS
#     INSTRUCTION-AWARE E5 RUN
# ================================================================

expected_aggregates = {

    "Exact@1":
        (
            0.002959,
            0.004438
        ),

    "TokenF1@1":
        (
            0.400414,
            0.367864
        ),

    "MeanCos@1(QSim)":
        (
            0.960814,
            0.967879
        ),

    "Semantic@1(ans_cos≥0.85)":
        (
            0.991124,
            0.982249
        ),

    "BERTScoreF1@1":
        (
            0.810350,
            0.800921
        ),
}


print(
    "\n"
    + "=" * 78
)

print(
    "AGGREGATE REPRODUCTION CHECK"
)

print(
    "=" * 78
)


core_match = True


for metric, (
    expected_base,
    expected_seg
) in expected_aggregates.items():

    actual_base = (
        base_summary[metric]
    )

    actual_seg = (
        seg_summary[metric]
    )


    tolerance = (
        5e-5
        if metric
        == "BERTScoreF1@1"
        else 5e-6
    )


    base_ok = np.isclose(
        actual_base,
        expected_base,
        atol=tolerance,
        rtol=0
    )

    seg_ok = np.isclose(
        actual_seg,
        expected_seg,
        atol=tolerance,
        rtol=0
    )


    print(
        f"{metric:<30} "
        f"BASE={actual_base:.6f} "
        f"SEG={actual_seg:.6f} "
        f"{'✅' if base_ok and seg_ok else '⚠️'}"
    )


    if (
        metric
        != "BERTScoreF1@1"
        and
        not (
            base_ok
            and seg_ok
        )
    ):

        core_match = False


if not core_match:

    raise RuntimeError(
        "\nSTOP: core E5 non-FT aggregate "
        "metrics do not reproduce the "
        "previous instruction-aware run."
    )


print(
    "\n✅ Core E5 results reproduce "
    "the instruction-aware run."
)


# ================================================================
# 21. STATISTICAL FUNCTIONS
# ================================================================

def paired_arrays(
    base_values,
    seg_values
):

    b = np.asarray(
        base_values,
        dtype=float
    ).reshape(-1)

    s = np.asarray(
        seg_values,
        dtype=float
    ).reshape(-1)


    if (
        b.shape != s.shape
        or
        b.size == 0
    ):

        raise ValueError(
            "Invalid paired arrays."
        )


    if (
        not np.all(
            np.isfinite(b)
        )
        or
        not np.all(
            np.isfinite(s)
        )
    ):

        raise ValueError(
            "NaN/Inf in paired arrays."
        )


    return b, s


def paired_bootstrap_ci(
    base_values,
    seg_values,
    n_boot=N_BOOT,
    alpha=STAT_ALPHA,
    seed=STAT_SEED
):

    b, s = paired_arrays(
        base_values,
        seg_values
    )

    diff = s - b

    observed_delta = float(
        np.mean(diff)
    )

    rng = np.random.default_rng(
        seed
    )

    n = len(diff)

    bootstrap_deltas = np.empty(
        n_boot,
        dtype=float
    )


    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        bootstrap_deltas[i] = (
            float(
                np.mean(
                    diff[idx]
                )
            )
        )


    ci_low, ci_high = np.quantile(
        bootstrap_deltas,
        [
            alpha / 2,
            1 - alpha / 2
        ]
    )


    return (
        observed_delta,
        float(ci_low),
        float(ci_high)
    )


def paired_signflip_p(
    base_values,
    seg_values,
    n_perm=N_PERM,
    seed=STAT_SEED
):

    b, s = paired_arrays(
        base_values,
        seg_values
    )

    diff = s - b


    if np.allclose(
        diff,
        0.0
    ):

        return 1.0


    observed = abs(
        float(
            np.mean(diff)
        )
    )


    rng = np.random.default_rng(
        seed
    )

    extreme = 0


    for _ in range(
        n_perm
    ):

        signs = rng.choice(
            np.array(
                [-1.0, 1.0]
            ),
            size=len(diff)
        )

        permuted = abs(
            float(
                np.mean(
                    diff * signs
                )
            )
        )


        if (
            permuted
            >= observed - 1e-15
        ):

            extreme += 1


    return float(
        (extreme + 1)
        /
        (n_perm + 1)
    )


def exact_mcnemar(
    base_values,
    seg_values
):

    b = np.asarray(
        base_values,
        dtype=int
    )

    s = np.asarray(
        seg_values,
        dtype=int
    )


    # BASE success, SEG failure
    n10 = int(
        np.sum(
            (b == 1)
            &
            (s == 0)
        )
    )


    # BASE failure, SEG success
    n01 = int(
        np.sum(
            (b == 0)
            &
            (s == 1)
        )
    )


    discordant = (
        n10 + n01
    )


    if discordant == 0:

        return (
            n10,
            n01,
            1.0
        )


    p = binomtest(
        n10,
        n=discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue


    return (
        n10,
        n01,
        float(p)
    )


def holm_adjust(
    p_values
):

    p = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p)

    order = np.argsort(p)

    adjusted_sorted = np.empty(
        m,
        dtype=float
    )

    running_max = 0.0


    for rank, idx in enumerate(
        order
    ):

        candidate = (
            (m - rank)
            *
            p[idx]
        )

        running_max = max(
            running_max,
            candidate
        )

        adjusted_sorted[rank] = min(
            1.0,
            running_max
        )


    adjusted = np.empty(
        m,
        dtype=float
    )


    for rank, idx in enumerate(
        order
    ):

        adjusted[idx] = (
            adjusted_sorted[rank]
        )


    return adjusted


# ================================================================
# 22. PRIMARY PAIRED INFERENCE
# ================================================================

metric_specs = [

    (
        "Exact@1",
        "Exact",
        "binary"
    ),

    (
        "TokenF1@1",
        "TokenF1",
        "continuous"
    ),

    (
        "MeanCos@1(QSim)",
        "QSim",
        "continuous"
    ),

    (
        "Semantic@1(ans_cos≥0.85)",
        "SemHit",
        "binary"
    ),

    (
        "BERTScoreF1@1",
        "BERTScoreF1",
        "continuous"
    ),
]


stat_rows = []


for j, (
    metric,
    key,
    metric_type
) in enumerate(
    metric_specs,
    start=1
):

    b = np.asarray(
        [
            r[key]
            for r in base_details
        ],
        dtype=float
    )

    s = np.asarray(
        [
            r[key]
            for r in seg_details
        ],
        dtype=float
    )


    delta, ci_low, ci_high = (
        paired_bootstrap_ci(
            b,
            s,
            n_boot=N_BOOT,
            alpha=STAT_ALPHA,
            seed=STAT_SEED + j
        )
    )


    n10 = np.nan
    n01 = np.nan


    if metric_type == "binary":

        (
            n10,
            n01,
            p_raw
        ) = exact_mcnemar(
            b.astype(int),
            s.astype(int)
        )

        test_name = (
            "Exact two-sided McNemar"
        )


    else:

        p_raw = paired_signflip_p(
            b,
            s,
            n_perm=N_PERM,
            seed=(
                STAT_SEED
                + 1000
                + j
            )
        )

        test_name = (
            "Two-sided paired "
            "sign-flip permutation"
        )


    stat_rows.append(
        {
            "Model":
                MODEL_NAME,

            "Stage":
                "NON_FINE_TUNED_STRICT_FAIR",

            "Metric":
                metric,

            "N":
                len(b),

            "Baseline":
                float(
                    np.mean(b)
                ),

            "Segmented":
                float(
                    np.mean(s)
                ),

            "Delta_SegMinusBase":
                delta,

            "CI95_low":
                ci_low,

            "CI95_high":
                ci_high,

            "p_raw":
                p_raw,

            "Test":
                test_name,

            "n10_Base1_Seg0":
                n10,

            "n01_Base0_Seg1":
                n01,
        }
    )


stats_df = pd.DataFrame(
    stat_rows
)


stats_df[
    "p_Holm"
] = holm_adjust(
    stats_df[
        "p_raw"
    ].to_numpy()
)


stats_df[
    "Significant_Holm"
] = (
    stats_df[
        "p_Holm"
    ]
    <
    STAT_ALPHA
)


stats_df[
    "Direction"
] = np.where(

    stats_df[
        "Delta_SegMinusBase"
    ] > 0,

    "Segmented higher",

    np.where(

        stats_df[
            "Delta_SegMinusBase"
        ] < 0,

        "Segmented lower",

        "No difference"
    )
)


stats_df[
    "CI95"
] = stats_df.apply(

    lambda r:
    (
        f"[{r['CI95_low']:.6f}, "
        f"{r['CI95_high']:.6f}]"
    ),

    axis=1
)


# ================================================================
# 23. PRINT STATISTICAL RESULTS
# ================================================================

display_columns = [

    "Metric",

    "N",

    "Baseline",

    "Segmented",

    "Delta_SegMinusBase",

    "CI95",

    "p_raw",

    "p_Holm",

    "Significant_Holm",

    "Direction",
]


print(
    "\n\n"
    + "=" * 78
)

print(
    "TABLE 2. E5 INSTRUCTION-AWARE "
    "NON-FT PAIRED INFERENCE"
)

print(
    "=" * 78
)


print(
    stats_df[
        display_columns
    ].to_string(
        index=False
    )
)


# ================================================================
# 24. MCNEMAR DETAILS
# ================================================================

print(
    "\n"
    "McNemar discordant-pair counts:"
)


mcnemar_df = stats_df[
    stats_df[
        "Metric"
    ].isin(
        [
            "Exact@1",
            "Semantic@1(ans_cos≥0.85)"
        ]
    )
][
    [
        "Metric",
        "n10_Base1_Seg0",
        "n01_Base0_Seg1",
        "p_raw",
        "p_Holm",
    ]
]


print(
    mcnemar_df.to_string(
        index=False
    )
)


# ================================================================
# 25. SAVE ITEM-LEVEL RESULTS
# ================================================================

base_details_path = (
    OUT_DIR /
    "nonft_base_details.csv"
)

seg_details_path = (
    OUT_DIR /
    "nonft_seg_details.csv"
)

summary_path = (
    OUT_DIR /
    "e5_instructionaware_nonft_summary.csv"
)

stats_path = (
    OUT_DIR /
    "e5_instructionaware_nonft_paired_statistics.csv"
)


pd.DataFrame(
    base_details
).to_csv(
    base_details_path,
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    seg_details
).to_csv(
    seg_details_path,
    index=False,
    encoding="utf-8-sig"
)


summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)


stats_df.to_csv(
    stats_path,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 26. ZIP OUTPUTS
# ================================================================

zip_path = Path(
    "/content/"
    "e5_instructionaware_nonft_"
    "recomputed_outputs.zip"
)


with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in OUT_DIR.iterdir():

        if p.is_file():

            z.write(
                p,
                arcname=p.name
            )


# ================================================================
# 27. FINAL COPY BLOCK
# ================================================================

print(
    "\n\n"
    + "=" * 78
)

print(
    "COPY THIS BLOCK BACK TO CHATGPT"
)

print(
    "=" * 78
)


print(
    f"Model = {MODEL_NAME}"
)

print(
    "Protocol = instruction-aware "
    "NON-FT strict-fair recomputed "
    "directly from JSON"
)

print(
    f"Clean records = {len(base_rows)}"
)

print(
    f"Segmented records = {len(seg_rows)}"
)

print(
    f"Clean unique = {len(base_map)}"
)

print(
    f"Segmented unique = {len(seg_map)}"
)

print(
    f"Paired total = {len(paired)}"
)

print(
    f"Train = {len(train_rows)}"
)

print(
    f"Test = {len(test_rows)}"
)

print(
    f"Seed = {SEED}"
)

print(
    f"Split SHA256 = {split_hash}"
)

print(
    "E5 query format = "
    "Instruct: Given a question in Kazakh, "
    "retrieve the most relevant answer. "
    "Query: <question>"
)

print(
    "Answer format = plain CLEAN answer"
)

print(
    f"BERTScore language = {bert_lang}"
)

print(
    "Delta = SEG - BASE"
)

print(
    "Continuous tests = "
    "paired two-sided sign-flip "
    "permutation, 10,000"
)

print(
    "Binary tests = "
    "exact two-sided McNemar"
)

print(
    "95% CI = "
    "paired bootstrap, 10,000"
)

print(
    "Multiple testing = "
    "Holm correction across 5 metrics"
)


print(
    "\nSUMMARY:"
)

print(
    summary_df.to_string(
        index=False
    )
)


print(
    "\nPAIRED STATISTICS:"
)

print(
    stats_df[
        display_columns
    ].to_string(
        index=False
    )
)


print(
    "\nMcNemar counts:"
)

print(
    mcnemar_df.to_string(
        index=False
    )
)


print(
    "\nFILES SAVED:"
)

print(base_details_path)
print(seg_details_path)
print(summary_path)
print(stats_path)
print(zip_path)


print(
    "\n✅ DONE"
)

E5-LARGE-INSTRUCT NON-FT STRICT-FAIR RECOMPUTATION
Device: cuda
GPU: Tesla T4
Model: intfloat/multilingual-e5-large-instruct
E5 query formatting: instruction-aware
Answer/passages: plain CLEAN text

================ FILES / RAW COUNTS ================
BASE: /content/baseline_15000.json | N=14,991
SEG : /content/kazakh_segmented_15000.json | N=14,998

================ STRICT PAIRING AUDIT ================
Clean records                  = 14,991
Segmented records              = 14,998
Clean unique keys              = 14,689
Segmented unique keys          = 14,696
Strict-fair paired keys        = 6,759
Clean duplicates collapsed     = 302
Segmented duplicates collapsed = 302
✅ Strict-fair pairing counts reproduced exactly.

================ STRICT-FAIR SPLIT ================
Total paired = 6,759
Train        = 6,083
Test         = 676
Seed         = 42
Split SHA256 = 867d3305fbc71068afedeae71b4ef10221968d23270202bf0eda860fc8bf880b
✅ Exact same 6083/676 split reproduced.
✅ CLEAN candidate/

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScore language used: kk


TABLE 1. E5 INSTRUCTION-AWARE NON-FT STRICT-FAIR RESULTS
 Stage      Condition   N  Exact@1  TokenF1@1  MeanCos@1(QSim)  Semantic@1(ans_cos≥0.85)  BERTScoreF1@1
NON_FT     BASE_CLEAN 676 0.002959   0.400414         0.960814                  0.991124       0.810350
NON_FT SEG_MORPH_ONLY 676 0.004438   0.367864         0.967879                  0.982249       0.800921

AGGREGATE REPRODUCTION CHECK
Exact@1                        BASE=0.002959 SEG=0.004438 ✅
TokenF1@1                      BASE=0.400414 SEG=0.367864 ✅
MeanCos@1(QSim)                BASE=0.960814 SEG=0.967879 ✅
Semantic@1(ans_cos≥0.85)       BASE=0.991124 SEG=0.982249 ✅
BERTScoreF1@1                  BASE=0.810350 SEG=0.800921 ✅

✅ Core E5 results reproduce the instruction-aware run.


TABLE 2. E5 INSTRUCTION-AWARE NON-FT PAIRED INFERENCE
                  Metric   N  Baseline  Segmented  Delta_SegMinusBase                   CI95    p_raw   p_Holm  Significant_Holm        Direction
           